In [9]:
import pandas as pd
import numpy as np

from pathlib import Path

PROJECT_DIR = Path.cwd().parent
DATA_PATH = PROJECT_DIR / "data" / "ola_rides_raw.xlsx"

df = pd.read_excel(DATA_PATH, engine='openpyxl')

clean_df = df.copy()

print("Original dataset:", df.shape)

Original dataset: (103024, 20)


In [11]:
clean_df.drop(
    columns=['Vehicle Images'],
    inplace=True,
    errors='ignore'
)

print("Updated shape:", clean_df.shape)

C:\Users\Rinku Saroj\AppData\Local\Temp\ipykernel_9976\90437408.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  clean_df.drop(


Updated shape: (103024, 19)


In [12]:
clean_df.columns = (
    clean_df.columns
    .str.strip()
    .str.replace(' ', '_')
)

print(clean_df.columns.tolist())

['Date', 'Time', 'Booking_ID', 'Booking_Status', 'Customer_ID', 'Vehicle_Type', 'Pickup_Location', 'Drop_Location', 'V_TAT', 'C_TAT', 'Canceled_Rides_by_Customer', 'Canceled_Rides_by_Driver', 'Incomplete_Rides', 'Incomplete_Rides_Reason', 'Booking_Value', 'Payment_Method', 'Ride_Distance', 'Driver_Ratings', 'Customer_Rating']


In [10]:
original_df = pd.read_excel(
    DATA_PATH,
    engine='openpyxl'
)

print(original_df['Time'].head(10))

print("\nData type:")
print(original_df['Time'].dtype)

print("\nSample values:")
print(original_df['Time'].dropna().astype(str).unique()[:15])

0    0.583333
1    0.930556
2    0.832639
3    0.135417
4    0.376389
5    0.195833
6    0.410417
7    0.466667
8    0.804861
9    0.063194
Name: Time, dtype: float64

Data type:
float64

Sample values:
<ArrowStringArray>
[ '0.583333333333333',  '0.930555555555556',  '0.832638888888889',
  '0.135416666666667',  '0.376388888888889',  '0.195833333333333',
  '0.410416666666667',  '0.466666666666667',  '0.804861111111111',
 '0.0631944444444444',            '0.98125',            '0.16875',
  '0.554166666666667',  '0.788194444444444',             '0.4125']
Length: 15, dtype: str


In [18]:
# Convert Excel decimal time into a proper time format

time_datetime = (
    pd.Timestamp('1970-01-01')
    + pd.to_timedelta(original_df['Time'], unit='D')
)

clean_df['Time'] = time_datetime.dt.strftime('%H:%M:%S')

display(clean_df[['Date', 'Time']].head(10))

,Date,Time
0,2024-07-26,13:59:59
1,2024-07-25,22:20:00
2,2024-07-30,19:59:00
3,2024-07-22,03:15:00
4,2024-07-02,09:02:00
5,2024-07-13,04:41:59
6,2024-07-23,09:51:00
7,2024-07-11,11:12:00
8,2024-07-01,19:18:59
9,2024-07-18,01:30:59


In [19]:
clean_df['Booking_Hour'] = pd.to_datetime(
    clean_df['Time'],
    format='%H:%M:%S'
).dt.hour

clean_df['Booking_Day'] = clean_df['Date'].dt.day

clean_df['Day_Name'] = clean_df['Date'].dt.day_name()

clean_df['Booking_Month'] = clean_df['Date'].dt.month_name()

clean_df['Ride_Time_Category'] = (
    clean_df['Booking_Hour'].apply(categorize_time)
)

display(
    clean_df[
        ['Date', 'Time', 'Booking_Hour', 'Ride_Time_Category']
    ].head(10)
)

,Date,Time,Booking_Hour,Ride_Time_Category
0,2024-07-26,13:59:59,13,Afternoon
1,2024-07-25,22:20:00,22,Night
2,2024-07-30,19:59:00,19,Evening
3,2024-07-22,03:15:00,3,Night
4,2024-07-02,09:02:00,9,Morning
5,2024-07-13,04:41:59,4,Night
6,2024-07-23,09:51:00,9,Morning
7,2024-07-11,11:12:00,11,Morning
8,2024-07-01,19:18:59,19,Evening
9,2024-07-18,01:30:59,1,Night


In [20]:
print("Missing Time:", clean_df['Time'].isna().sum())

print(
    "Missing Booking Hour:",
    clean_df['Booking_Hour'].isna().sum()
)

print("\nBookings by hour:")
display(clean_df['Booking_Hour'].value_counts().sort_index())

print("\nBookings by time category:")
display(clean_df['Ride_Time_Category'].value_counts())

Missing Time: 0
Missing Booking Hour: 0

Bookings by hour:


Booking_Hour
0     4318
1     4392
2     4242
3     4201
4     4330
5     4197
6     4271
7     4388
8     4290
9     4347
10    4409
11    4197
12    4408
13    4363
14    4154
15    4376
16    4344
17    4270
18    4176
19    4356
20    4143
21    4343
22    4362
23    4147
Name: count, dtype: int64


Bookings by time category:


Ride_Time_Category
Night        34335
Morning      30099
Afternoon    21645
Evening      16945
Name: count, dtype: int64

In [21]:
OUTPUT_PATH = PROJECT_DIR / "data" / "ola_rides_cleaned.csv"

clean_df.to_csv(OUTPUT_PATH, index=False)

print("Final cleaned dataset saved!")
print("Shape:", clean_df.shape)

Final cleaned dataset saved!
Shape: (103024, 24)


In [22]:
print("Missing Time:", clean_df['Time'].isna().sum())
print("Missing Booking Hour:", clean_df['Booking_Hour'].isna().sum())

print("\nFirst 10 records:")
display(
    clean_df[['Date', 'Time', 'Booking_Hour', 'Ride_Time_Category']].head(10)
)

print("\nTime category counts:")
display(clean_df['Ride_Time_Category'].value_counts())

print("\nMinimum hour:", clean_df['Booking_Hour'].min())
print("Maximum hour:", clean_df['Booking_Hour'].max())

Missing Time: 0
Missing Booking Hour: 0

First 10 records:


,Date,Time,Booking_Hour,Ride_Time_Category
0,2024-07-26,13:59:59,13,Afternoon
1,2024-07-25,22:20:00,22,Night
2,2024-07-30,19:59:00,19,Evening
3,2024-07-22,03:15:00,3,Night
4,2024-07-02,09:02:00,9,Morning
5,2024-07-13,04:41:59,4,Night
6,2024-07-23,09:51:00,9,Morning
7,2024-07-11,11:12:00,11,Morning
8,2024-07-01,19:18:59,19,Evening
9,2024-07-18,01:30:59,1,Night



Time category counts:


Ride_Time_Category
Night        34335
Morning      30099
Afternoon    21645
Evening      16945
Name: count, dtype: int64


Minimum hour: 0
Maximum hour: 23
